# `mamba_chunk_scan_combined` — Annotated Forward Pass

This notebook traces through every line of the forward pass, explaining what each
sub-kernel computes, with concrete shapes and the underlying math.

## The SSM recurrence (what we're computing)

The continuous SSM is:
```
h'(t) = A * h(t) + B(t) * x(t)
y(t)  = C(t) * h(t) + D * x(t)
```

After discretization with step size `dt`, for each time step `t`:
```
h[t] = exp(A * dt[t]) * h[t-1] + dt[t] * B[t] * x[t]
y[t] = C[t] @ h[t] + D * x[t]
```

**Mamba2 key insight**: A is a **scalar per head** (not a matrix), so `exp(A * dt)` is
just a scalar decay. This makes the recurrence a **scalar-times-matrix** update, which
enables efficient chunked parallel computation via the SSD (Structured State-space Duality)
algorithm.

## Notation used throughout

| Symbol | Name | Typical value |
|--------|------|---------------|
| `B` (batch) | batch size | 2 |
| `L` (seqlen) | sequence length | 512 |
| `H` (nheads) | number of SSM heads | 24 |
| `P` (headdim) | dimension per head | 64 |
| `N` (dstate) | SSM state dimension | 128 |
| `G` (ngroups) | groups for B,C sharing | 1 |
| `Q` (chunk_size) | chunk length | 256 |
| `K` (nchunks) | ceil(L/Q) | 2 |

In [ ]:
import os, sys, types, math
os.environ.setdefault("CC", os.path.expanduser("~/miniconda3/envs/cutedsl/bin/x86_64-conda-linux-gnu-gcc"))
MAMBA_ROOT = os.path.expanduser("~/mamba")
sys.path.insert(0, MAMBA_ROOT)
pkg = types.ModuleType("mamba_ssm")
pkg.__path__ = [os.path.join(MAMBA_ROOT, "mamba_ssm")]
pkg.__package__ = "mamba_ssm"
sys.modules["mamba_ssm"] = pkg

In [ ]:
import torch
import torch.nn.functional as F
from einops import rearrange

# Import the individual sub-kernels so we can call them one by one
from mamba_ssm.ops.triton.ssd_chunk_state import _chunk_cumsum_fwd, _chunk_state_fwd
from mamba_ssm.ops.triton.ssd_state_passing import _state_passing_fwd
from mamba_ssm.ops.triton.ssd_bmm import _bmm_chunk_fwd
from mamba_ssm.ops.triton.ssd_chunk_scan import _chunk_scan_fwd
# And the combined function to verify our step-by-step matches
from mamba_ssm.ops.triton.ssd_combined import mamba_chunk_scan_combined

torch.set_printoptions(precision=4, sci_mode=False)
device = "cuda"
dtype = torch.float32

In [ ]:
# ---- Dimensions ----
B_dim    = 2       # batch
L        = 512     # seqlen
H        = 24      # nheads
P        = 64      # headdim
N        = 128     # dstate
G        = 1       # ngroups  (B,C are shared across H//G heads each)
Q        = 256     # chunk_size
K_chunks = math.ceil(L / Q)  # nchunks = 2

print(f"nchunks = ceil({L}/{Q}) = {K_chunks}")
print(f"heads_per_group = {H // G}")

In [ ]:
# ---- Create inputs ----
torch.manual_seed(42)

x       = torch.randn(B_dim, L, H, P, device=device, dtype=dtype)    # (B, L, H, P)
dt_in   = torch.randn(B_dim, L, H, device=device, dtype=dtype)       # (B, L, H)
A       = -torch.rand(H, device=device, dtype=dtype)                  # (H,)  MUST be negative
B_mat   = torch.randn(B_dim, L, G, N, device=device, dtype=dtype)    # (B, L, G, N)
C_mat   = torch.randn(B_dim, L, G, N, device=device, dtype=dtype)    # (B, L, G, N)
D_vec   = torch.randn(H, device=device, dtype=dtype)                 # (H,)
dt_bias = torch.randn(H, device=device, dtype=dtype)                 # (H,)
z       = None  # optional gating, skip for clarity

print("Inputs:")
for name, t in [("x", x), ("dt", dt_in), ("A", A), ("B", B_mat), ("C", C_mat), ("D", D_vec), ("dt_bias", dt_bias)]:
    print(f"  {name:8s}: {str(list(t.shape)):30s} {t.dtype}")

---
## Overview: The 5-stage pipeline

The forward pass decomposes the chunked SSM computation into 5 kernels:

```
                      dt, A, dt_bias
                          |
                   [1] _chunk_cumsum_fwd
                          |
                    dA_cumsum, dt_processed
                    /            \
        B, x, dt, dA_cumsum     C, B
              |                    |
     [2] _chunk_state_fwd   [4] _bmm_chunk_fwd
              |                    |
          states_raw              CB
              |                    \
     [3] _state_passing_fwd        \
              |                     \
          states_propagated          \
              \                      /
               \                    /
            [5] _chunk_scan_fwd
                    |
                   out
```

Note: Steps [2] and [4] are independent and could run in parallel.
Step [3] depends on [2]. Step [5] depends on [3] and [4].

---
## Step 1: `_chunk_cumsum_fwd` — Compute per-step decay

**File**: `ssd_chunk_state.py:718`  
**Triton kernel**: `_chunk_cumsum_fwd_kernel` (line 40)

### What it does (pseudocode)
```python
# For each batch b, head h, chunk c, position i within chunk:
dt_eff[b,h,c,i] = softplus(dt[b, c*Q+i, h] + dt_bias[h])   # if dt_softplus
dt_eff[b,h,c,i] = clamp(dt_eff[b,h,c,i], dt_min, dt_max)
dA[b,h,c,i]     = dt_eff[b,h,c,i] * A[h]                   # scalar decay per step
dA_cumsum[b,h,c,i] = cumsum(dA[b,h,c,:], dim=i)             # cumsum WITHIN each chunk
```

### Key insight
- `dA_cumsum[b,h,c,i]` = total log-decay from position 0 to position i within chunk c.
- So `exp(dA_cumsum[b,h,c,j] - dA_cumsum[b,h,c,i])` = decay from position i to j (where j >= i).
- Since A is negative, dA is negative, so dA_cumsum is monotonically decreasing (more decay over time).
- The cumsum is reset at each chunk boundary.

In [ ]:
# ========== STEP 1: _chunk_cumsum_fwd ==========
#
# Input:   dt        (B, L, H)         -- raw time-step values
#          A         (H,)              -- state decay rates (negative)
#          dt_bias   (H,)              -- bias added before softplus
#
# Output:  dA_cumsum (B, H, K, Q)     -- cumulative log-decay within each chunk
#          dt_out    (B, H, K, Q)     -- processed dt values (after bias + softplus + clamp)
#
# where K = nchunks = ceil(L/Q)

dA_cumsum, dt_out = _chunk_cumsum_fwd(
    dt_in, A, Q,
    dt_bias=dt_bias,
    dt_softplus=True,
    dt_limit=(0.0, float("inf")),
)

print(f"dA_cumsum: {list(dA_cumsum.shape)}  = (B={B_dim}, H={H}, K={K_chunks}, Q={Q})")
print(f"dt_out:    {list(dt_out.shape)}  = (B={B_dim}, H={H}, K={K_chunks}, Q={Q})")
print(f"\ndA_cumsum dtype: {dA_cumsum.dtype}  (always fp32 for numerical stability)")
print(f"dt_out dtype:    {dt_out.dtype}")

# Verify: dA_cumsum should be negative and decreasing within each chunk
print(f"\ndA_cumsum[0, 0, 0, :8] = {dA_cumsum[0, 0, 0, :8]}")
print(f"dt_out[0, 0, 0, :8]    = {dt_out[0, 0, 0, :8]}")
print(f"A[0] = {A[0]:.4f}  (negative => dA_cumsum decreases)")

In [ ]:
# ---- Manual verification of Step 1 ----
# Let's verify for head 0, chunk 0, first few positions

h_idx = 0
dt_raw = dt_in[0, :Q, h_idx]                              # (Q,) raw dt for chunk 0
dt_manual = F.softplus(dt_raw + dt_bias[h_idx])            # add bias, softplus
dt_manual = dt_manual.clamp(min=0.0)                       # clamp to [0, inf)
dA_manual = dt_manual * A[h_idx]                           # multiply by A (negative)
dA_cs_manual = torch.cumsum(dA_manual, dim=0)              # cumsum within chunk

print("Manual vs kernel (first 8 positions, head 0, chunk 0):")
print(f"  dt_manual:    {dt_manual[:8]}")
print(f"  dt_kernel:    {dt_out[0, h_idx, 0, :8]}")
print(f"  dA_cs_manual: {dA_cs_manual[:8]}")
print(f"  dA_cs_kernel: {dA_cumsum[0, h_idx, 0, :8]}")
print(f"  Match: {torch.allclose(dA_cs_manual, dA_cumsum[0, h_idx, 0, :Q], atol=1e-5)}")

---
## Step 2: `_chunk_state_fwd` — Compute per-chunk boundary states

**File**: `ssd_chunk_state.py:812`  
**Triton kernel**: `_chunk_state_fwd_kernel` (line 192)

### What it does

For each chunk `c`, this computes the **SSM state at the END of the chunk** assuming
the state at the START of the chunk was zero. (The actual initial state is handled in Step 3.)

Mathematically, the state at the end of chunk c (with zero initial state) is:
```
state[c, h, p, n] = sum over i in [0..Q-1] of:
    exp(dA_cumsum[c, Q-1] - dA_cumsum[c, i])   # decay from position i to end of chunk
    * dt[c, i]                                   # discretization step
    * x[c, i, h, p]                              # input at position i
    * B[c, i, g, n]                              # state input matrix
```

This is a matrix multiply: `states[p,n] = sum_i (decay_i * dt_i * x[i,p]) * B[i,n]`

### The Triton kernel in detail (lines 238-268)
```python
# For each (batch, chunk, head), parallelized over (headdim_block, dstate_block):
dA_cs_last = dA_cumsum[..., chunk_size - 1]      # log-decay at end of chunk
acc = zeros(BLOCK_SIZE_M, BLOCK_SIZE_N)           # (headdim_block, dstate_block)
for k in range(0, chunk_size, BLOCK_SIZE_K):
    scale = exp(min(dA_cs_last - dA_cs[k], 0)) * dt[k]  # decay from k to end * dt
    b_scaled = B[k, :] * scale                    # (dstate_block,) scaled by decay*dt
    acc += x[k, :, None] @ b_scaled[None, :]      # outer product, accumulated
states = acc                                       # (headdim, dstate)
```

In [ ]:
# ========== STEP 2: _chunk_state_fwd ==========
#
# Input:   B_mat      (B, L, G, N)          -- SSM state input matrix
#          x          (B, L, H, P)          -- input sequence
#          dt_out     (B, H, K, Q)          -- processed dt from Step 1
#          dA_cumsum  (B, H, K, Q)          -- cumulative decay from Step 1
#
# Output:  states     (B, K, H, P, N)      -- per-chunk boundary state
#                                              (state at end of chunk, assuming zero initial)

states_raw = _chunk_state_fwd(
    B_mat, x, dt_out, dA_cumsum,
    seq_idx=None,
    states_in_fp32=True,
)

print(f"states_raw: {list(states_raw.shape)}  = (B={B_dim}, K={K_chunks}, H={H}, P={P}, N={N})")
print(f"dtype: {states_raw.dtype}  (fp32 for numerical stability)")
print(f"\nInterpretation: states_raw[b, c, h, :, :] is the (P x N) state matrix")
print(f"at the end of chunk c for head h, assuming the chunk started with zero state.")

In [ ]:
# ---- Manual verification of Step 2 ----
# Verify for batch=0, chunk=0, head=0
# NOTE: Small diffs (~1e-2) are expected because the Triton kernel accumulates
# the matmul in tiled blocks with a different summation order than our naive loop.

b, c, h = 0, 0, 0
g = h // (H // G)  # which group this head belongs to

# Get the chunk's data
x_chunk  = x[b, c*Q:(c+1)*Q, h, :]           # (Q, P)
B_chunk  = B_mat[b, c*Q:(c+1)*Q, g, :]        # (Q, N)
dt_chunk = dt_out[b, h, c, :]                  # (Q,)
dA_cs    = dA_cumsum[b, h, c, :]               # (Q,)

# decay from each position to end of chunk
dA_cs_last = dA_cs[Q-1]
# Use min(., 0) for numerical stability (same as kernel)
decay_to_end = torch.exp(torch.minimum(dA_cs_last - dA_cs, torch.zeros_like(dA_cs)))  # (Q,)
scale = decay_to_end * dt_chunk                                                         # (Q,)

# state = sum_i scale[i] * outer(x[i], B[i])
# = (x * scale[:, None]).T @ B   -- matrix multiply (P,Q) @ (Q,N) = (P,N)
state_manual = (x_chunk * scale[:, None]).T @ B_chunk  # (P, N)

diff = (state_manual - states_raw[b, c, h]).abs().max().item()
print(f"Max diff (manual vs kernel): {diff:.2e}  (fp32 accumulation order differences)")
print(f"Correlation: {torch.corrcoef(torch.stack([state_manual.flatten(), states_raw[b, c, h].flatten()]))[0,1]:.6f}")

---
## Step 3: `_state_passing_fwd` — Propagate states across chunks

**File**: `ssd_state_passing.py:196`  
**Triton kernel**: `_state_passing_fwd_kernel` (line 30)

### What it does

Step 2 gave us `states_raw[c]` = the state at the end of chunk c assuming zero initial state.
But the actual state at the end of chunk c depends on the state at the end of chunk c-1!

This kernel does a **sequential scan across chunks** to propagate states:
```
state_out[0] = initial_state               # (or zero)
state_out[1] = exp(dA_cs_last[0]) * state_out[0] + states_raw[0]
state_out[2] = exp(dA_cs_last[1]) * state_out[1] + states_raw[1]
...
final_state  = exp(dA_cs_last[K-1]) * state_out[K-1] + states_raw[K-1]
```

Where `dA_cs_last[c] = dA_cumsum[b, h, c, Q-1]` is the total log-decay over chunk c.

### The Triton kernel in detail (lines 64-87)
```python
# Sequential loop over chunks (this is inherently serial!)
states = initial_states  # or zeros
out[0] = states          # state at START of chunk 0
for c in range(nchunks):
    new_states = input_states[c]                 # raw state from Step 2
    scale = exp(dA_chunk_cumsum[c])               # decay over entire chunk c
    states = scale * states + new_states           # combine: decay old + add new
    if c < nchunks - 1:
        out[c+1] = states     # state at START of chunk c+1
    else:
        final_states = states  # state after all chunks
```

### Important output interpretation
- `out[c]` = state at the **START** of chunk c (i.e., end of chunk c-1, after propagation)
- `final_states` = state after the last chunk
- These will be used in Step 5 to compute outputs within each chunk

In [ ]:
# ========== STEP 3: _state_passing_fwd ==========
#
# Input:   states_raw     (B, K, H, P*N)   -- from Step 2, flattened last 2 dims
#          dA_chunk_last  (B, H, K)         -- total decay per chunk (last element of cumsum)
#          initial_states (B, H, P*N)       -- optional initial state (None here)
#
# Output:  states_out     (B, K, H, P*N)   -- state at START of each chunk
#          final_states   (B, H, P*N)       -- state after last chunk
#
# Note: the P*N flattening is because the scan is over a generic "dim" axis.
# The kernel doesn't care about the P,N structure — it just multiplies by a scalar.

# dA_cumsum[:, :, :, -1] = total log-decay for each chunk
dA_chunk_last = dA_cumsum[:, :, :, -1]  # (B, H, K)
print(f"dA_chunk_last: {list(dA_chunk_last.shape)}  = (B, H, K)")
print(f"dA_chunk_last[0, 0, :] = {dA_chunk_last[0, 0, :]}  (total decay per chunk, head 0)")

# Flatten (P, N) -> (P*N) for the state passing kernel
states_flat = rearrange(states_raw, "... p n -> ... (p n)")
print(f"\nstates_flat: {list(states_flat.shape)}  = (B, K, H, P*N={P*N})")

states_propagated, final_states = _state_passing_fwd(
    states_flat,
    dA_chunk_last,
    initial_states=None,  # zero initial state
    seq_idx=None,
    chunk_size=Q,
    out_dtype=C_mat.dtype,
)

# Unflatten back to (P, N)
states_propagated = rearrange(states_propagated, "... (p n) -> ... p n", n=N)
final_states = rearrange(final_states, "... (p n) -> ... p n", n=N)

print(f"\nstates_propagated: {list(states_propagated.shape)}  = (B, K, H, P, N)")
print(f"final_states:      {list(final_states.shape)}  = (B, H, P, N)")
print(f"\nInterpretation:")
print(f"  states_propagated[b, c] = SSM state at START of chunk c")
print(f"  final_states[b]         = SSM state after the last chunk")

In [ ]:
# ---- Manual verification of Step 3 ----

b, h = 0, 0

# Chunk 0: state at start = zero (no initial_states)
state_start_0 = torch.zeros(P, N, device=device, dtype=dtype)
print(f"state_start[chunk=0] is zero: {(states_propagated[b, 0, h] == 0).all()}")

# State at end of chunk 0 = decay * start + raw_state
decay_0 = torch.exp(dA_chunk_last[b, h, 0])  # scalar
state_end_0 = decay_0 * state_start_0 + states_raw[b, 0, h]  # = states_raw[0] since start=0

# This should equal state_start of chunk 1
diff = (state_end_0 - states_propagated[b, 1, h].float()).abs().max().item()
print(f"state_end[chunk=0] == state_start[chunk=1]: diff={diff:.2e}")

# State at end of chunk 1 = final_states
decay_1 = torch.exp(dA_chunk_last[b, h, 1])
state_end_1 = decay_1 * state_end_0 + states_raw[b, 1, h]
diff2 = (state_end_1 - final_states[b, h].float()).abs().max().item()
print(f"state_end[chunk=1] == final_states: diff={diff2:.2e}")

---
## Step 4: `_bmm_chunk_fwd` — Compute CB (intra-chunk attention)

**File**: `ssd_bmm.py:165`  
**Triton kernel**: `_bmm_chunk_fwd_kernel` (line 37)

### What it does

Computes `CB[b, c, g, i, j] = C[b, c*Q+i, g, :] @ B[b, c*Q+j, g, :]^T` for all `i, j` within each chunk.

This is a **batch of (Q x Q) matrices**, one per (batch, chunk, group).

```
CB[b, c, g, i, j] = sum_n C[b, c*Q+i, g, n] * B[b, c*Q+j, g, n]
```

### Why?

In the SSD formulation, the **intra-chunk** output computation involves:
```
y[i] += sum_{j<=i} C[i] @ exp(dA[i..j]) @ B[j]^T * dt[j] * x[j]
```

The `C @ B^T` part is the same regardless of `x`, so we precompute it as the CB matrix.
The decay and dt weighting are applied later in Step 5.

### Triton kernel (lines 74-89)
```python
# For each (batch, chunk, group), compute CB as matmul over dstate:
acc = zeros(BLOCK_M, BLOCK_N)  # (chunk_block_i, chunk_block_j)
for k in range(dstate / BLOCK_K):
    a = C[i, k*BK:(k+1)*BK]     # (BLOCK_M, BLOCK_K)
    b = B[k*BK:(k+1)*BK, j]     # (BLOCK_K, BLOCK_N)  -- transposed access
    acc += dot(a, b)
CB = acc
```

In [ ]:
# ========== STEP 4: _bmm_chunk_fwd ==========
#
# Input:   C_mat  (B, L, G, N)   -- SSM output matrix
#          B_mat  (B, L, G, N)   -- SSM input matrix
#          Q      (int)          -- chunk_size
#
# Output:  CB     (B, K, G, Q, Q)  -- intra-chunk "attention" matrix
#                                     CB[b,c,g,i,j] = C[b,c*Q+i,g,:] . B[b,c*Q+j,g,:]

CB = _bmm_chunk_fwd(
    C_mat, B_mat, Q,
    seq_idx=None,
    output_dtype=torch.float32,
)

print(f"CB: {list(CB.shape)}  = (B={B_dim}, K={K_chunks}, G={G}, Q={Q}, Q={Q})")
print(f"dtype: {CB.dtype}")
print(f"\nInterpretation: CB[b,c,g,i,j] = dot product of C[position i] and B[position j]")
print(f"within chunk c. This is a {Q}x{Q} matrix per (batch, chunk, group).")

In [ ]:
# ---- Manual verification of Step 4 ----
# NOTE: Small diffs (~1e-2) are expected from fp32 tiled accumulation vs naive matmul.
b, c, g = 0, 0, 0
C_chunk = C_mat[b, c*Q:(c+1)*Q, g, :]   # (Q, N)
B_chunk = B_mat[b, c*Q:(c+1)*Q, g, :]   # (Q, N)
CB_manual = C_chunk @ B_chunk.T          # (Q, Q)

diff = (CB_manual - CB[b, c, g]).abs().max().item()
print(f"Max diff (manual matmul vs kernel): {diff:.2e}  (fp32 accumulation order)")
print(f"Correlation: {torch.corrcoef(torch.stack([CB_manual.flatten(), CB[b, c, g].flatten()]))[0,1]:.6f}")

---
## Step 5: `_chunk_scan_fwd` — Final output computation

**File**: `ssd_chunk_scan.py:1259`  
**Triton kernel**: `_chunk_scan_fwd_kernel` (line 49)

### What it does

This is the main output kernel. For each position `i` in chunk `c`, it computes:

```
y[i] = (inter-chunk contribution) + (intra-chunk contribution) + (skip connection)
```

#### Part A: Inter-chunk contribution (lines 105-128)
The SSM state from previous chunks affects position i:
```python
# prev_state = states_propagated[c]   shape (P, N) -- state at START of chunk c
# C[i]                                 shape (N,)  -- output projection at position i
# dA_cumsum[c, i]                      scalar      -- decay from start of chunk to position i
#
# inter_chunk[i, p] = exp(dA_cumsum[c, i]) * sum_n C[i, n] * prev_state[p, n]
#                   = exp(dA_cumsum[c, i]) * (C[i] @ prev_state.T)     for each p
```
This is a matmul: `C (Q, N) @ prev_state (N, P) = (Q, P)`, scaled by `exp(dA_cumsum)`.

#### Part B: Intra-chunk contribution (lines 130-154)
Positions within the same chunk also contribute (the "diagonal block" in SSD):
```python
# For each position i, sum over positions j <= i in the same chunk:
# intra[i, p] = sum_j CB[i,j] * exp(dA_cumsum[i] - dA_cumsum[j]) * dt[j] * x[j, p]
#
# This is computed as a matmul:
#   L_matrix[i,j] = CB[i,j] * exp(dA_cs[i] - dA_cs[j]) * dt[j]   -- (Q, Q) matrix
#   intra = L_matrix @ x                                           -- (Q, Q) @ (Q, P) = (Q, P)
```
The `IS_CAUSAL=True` flag ensures j <= i (lower triangular).

#### Part C: Skip connection (lines 159-166)
```python
# skip[i, p] = D[h] * x[i, p]        if D is (H,)
# skip[i, p] = D[h, p] * x[i, p]     if D is (H, P)
```

#### Part D: Optional gating (lines 168-176)
If z is provided:
```python
out_x = inter + intra + skip           # save pre-gated output
out   = out_x * z * sigmoid(z)          # SiLU gating
```

In [ ]:
# ========== STEP 5: _chunk_scan_fwd ==========
#
# Input:   CB                (B, K, G, Q, Q)   -- from Step 4
#          x                 (B, L, H, P)      -- input
#          dt_out            (B, H, K, Q)      -- processed dt from Step 1
#          dA_cumsum         (B, H, K, Q)      -- cumulative decay from Step 1
#          C_mat             (B, L, G, N)      -- SSM output matrix
#          states_propagated (B, K, H, P, N)   -- from Step 3
#          D_vec             (H,) or (H,P)     -- skip connection
#          z                 None or (B,L,H,P) -- optional gating
#
# Output:  out               (B, L, H, P)      -- final output
#          out_x             (B, L, H, P)      -- pre-gated output (None if z is None)

out, out_x = _chunk_scan_fwd(
    CB, x, dt_out, dA_cumsum,
    C_mat, states_propagated,
    D=D_vec, z=z, seq_idx=None,
)

print(f"out:   {list(out.shape)}  = (B={B_dim}, L={L}, H={H}, P={P})")
print(f"out_x: {out_x}  (None because z is None)")
print(f"\nout[0, 0, 0, :8] = {out[0, 0, 0, :8]}")

In [ ]:
# ---- Manual verification of Step 5 ----
# Verify one position: batch=0, chunk=0, position=3 (within chunk), head=0
# NOTE: Small diffs from fp32 accumulation order in Triton tiled matmul.

b, c, h = 0, 0, 0
i = 3   # position within chunk
g = h // (H // G)
global_pos = c * Q + i

# Part A: Inter-chunk contribution
prev_state = states_propagated[b, c, h]  # (P, N)
C_i = C_mat[b, global_pos, g, :]         # (N,)
decay_from_start = torch.exp(dA_cumsum[b, h, c, i])  # scalar
inter_chunk = decay_from_start * (C_i @ prev_state.float().T)  # (P,)

# Part B: Intra-chunk contribution (sum over j <= i)
intra_chunk = torch.zeros(P, device=device)
for j in range(i + 1):  # j = 0, 1, 2, 3
    cb_ij = CB[b, c, g, i, j]  # C[i] . B[j]
    decay_ij = torch.exp(torch.minimum(
        dA_cumsum[b, h, c, i] - dA_cumsum[b, h, c, j],
        torch.tensor(0.0, device=device)
    ))
    dt_j = dt_out[b, h, c, j]
    x_j = x[b, global_pos - i + j, h, :]  # (P,)
    intra_chunk += cb_ij * decay_ij * dt_j * x_j

# Part C: Skip connection
skip = D_vec[h] * x[b, global_pos, h, :]  # (P,)

# Total
y_manual = inter_chunk + intra_chunk + skip

y_kernel = out[b, global_pos, h, :]
diff = (y_manual - y_kernel).abs().max().item()
print(f"Position {global_pos}, head {h}:")
print(f"  Inter-chunk norm: {inter_chunk.norm():.4f}  (zero for chunk 0 since no prior state)")
print(f"  Intra-chunk norm: {intra_chunk.norm():.4f}  (sum of j=0..{i} contributions)")
print(f"  Skip norm:        {skip.norm():.4f}  (D * x)")
print(f"  Max diff:         {diff:.2e}  (fp32 accumulation order)")
print(f"  Correlation:      {torch.corrcoef(torch.stack([y_manual, y_kernel]))[0,1]:.6f}")

---
## Verify: Our step-by-step matches `mamba_chunk_scan_combined`

In [ ]:
# Run the combined function
out_combined = mamba_chunk_scan_combined(
    x, dt_in, A, B_mat, C_mat,
    chunk_size=Q,
    D=D_vec,
    z=None,
    dt_bias=dt_bias,
    dt_softplus=True,
)

max_diff = (out - out_combined).abs().max().item()
print(f"Max diff (step-by-step vs combined): {max_diff:.2e}")
print(f"Match: {torch.allclose(out, out_combined, atol=1e-5)}")

---
## Summary: Complete shape trace

```
INPUTS:
  x       (B, L, H, P)       = (2, 512, 24, 64)
  dt      (B, L, H)          = (2, 512, 24)
  A       (H,)               = (24,)
  B       (B, L, G, N)       = (2, 512, 1, 128)
  C       (B, L, G, N)       = (2, 512, 1, 128)
  D       (H,)               = (24,)
  dt_bias (H,)               = (24,)

STEP 1: _chunk_cumsum_fwd
  dt + dt_bias -> softplus -> clamp -> dA = dt * A -> cumsum within chunk
  -> dA_cumsum  (B, H, K, Q)  = (2, 24, 2, 256)
  -> dt_out     (B, H, K, Q)  = (2, 24, 2, 256)

STEP 2: _chunk_state_fwd
  For each chunk: state = sum_i decay_to_end[i] * dt[i] * outer(x[i], B[i])
  -> states_raw (B, K, H, P, N) = (2, 2, 24, 64, 128)

STEP 3: _state_passing_fwd
  Sequential scan: state[c] = decay * state[c-1] + states_raw[c]
  -> states     (B, K, H, P, N) = (2, 2, 24, 64, 128)  -- state at START of each chunk
  -> final      (B, H, P, N)    = (2, 24, 64, 128)      -- state after last chunk

STEP 4: _bmm_chunk_fwd
  CB[i,j] = C[i] @ B[j]^T within each chunk
  -> CB     (B, K, G, Q, Q) = (2, 2, 1, 256, 256)

STEP 5: _chunk_scan_fwd
  y[i] = exp(dA_cs[i]) * C[i] @ state[c]          (inter-chunk)
       + sum_{j<=i} CB[i,j] * decay[i,j] * dt[j] * x[j]  (intra-chunk)
       + D * x[i]                                   (skip)
  -> out     (B, L, H, P) = (2, 512, 24, 64)
```